In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define dataset variable configurations
DATASET_VARS = {
    "BH_1": {
        "obj_var": "yield",
        "cat_vars": [
            "Aryl_halide_SMILES",
            "Additive_SMILES",
            "Base_SMILES",
            "Ligand_SMILES",
        ],
        "con_vars": [],
    },
    "DA": {
        "obj_var": "yield",
        "cat_vars": ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"],
        "con_vars": ["Concentration", "Temp_C"],
    },
    "alkox": {
        "obj_var": "conversion",
        "cat_vars": [],
        "con_vars": ["catalase", "peroxidase", "alcohol_oxidase", "ph"],
    },
    "oer_plate_a": {
        "obj_var": "overpotential",
        "cat_vars": [],
        "con_vars": ["ni_load", "fe_load", "co_load", "mn_load", "ce_load", "la_load"],
    },
    "p3ht": {
        "obj_var": "conductivity",
        "cat_vars": [],
        "con_vars": [
            "p3ht_content",
            "d1_content",
            "d2_content",
            "d6_content",
            "d8_content",
        ],
    },
    "photo_pce10": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "photo_wf3": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "suzuki_edbo": {
        "obj_var": "yield",
        "cat_vars": ["electrophile", "nucleophile", "base", "ligand", "solvent"],
        "con_vars": [],
    },
    "suzuki": {
        "obj_var": "yield",
        "cat_vars": [],
        "con_vars": ["temperature", "pd_mol", "arbpin", "k3po4"],
    },
}

In [ ]:
TIME_TAGS = [
    "20251106221726",
    "20251106221742",
    "20251106221802",
    "20251106223331",
    "20251106223413",
    "20251106223452",
]

## Processing

In [ ]:
# Read batch output logs
batch_output_logs_df = pd.read_csv("results_final/batch_output_logs.csv")

In [ ]:
exp_extracted_dfs = {}
pair_original_dfs = {}
pair_result_dfs = {}

for i, batch_output_logs_df_row in batch_output_logs_df[
    ["dataset_name", "model", "model_w_params", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    time_tag = batch_output_logs_df_row.time_tag

    # Read experimental data
    exp_extracted_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_extracted.csv"
    )
    exp_extracted_dfs[dataset_name] = exp_extracted_df

    # Read pair data
    pair_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_pair.csv"
    )
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    # Judge only by numerical magnitude
    pair_original_df["setup_true"] = ""
    pair_original_df.loc[
        pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
        "setup_true",
    ] = "A"
    pair_original_df.loc[
        pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
        "setup_true",
    ] = "B"
    assert len(pair_original_df.loc[pair_original_df["setup_true"] == ""]) == 0

    # Add features
    pair_original_df[f"{obj_var}_diff_abs"] = (
        pair_original_df[f"{obj_var}_A"] - pair_original_df[f"{obj_var}_B"]
    ).abs()
    pair_original_dfs[dataset_name] = pair_original_df

    # Read pair prediction results
    pair_result_df = pd.read_csv(
        Path("./results_final")
        / f"results_{time_tag}"
        / f"dataset_{dataset_name}_{model}_pair_result.csv"
    )
    pair_result_df = pair_result_df.rename(columns={"setup": "setup_predicted"})

    # For datasets where lower values are better, swap A and B to align numerical order
    if dataset_name in [
        "oer_plate_a",
        "photo_pce10",
        "photo_wf3",
    ]:
        pair_result_df["setup_predicted"] = pair_result_df["setup_predicted"].replace(
            {"A": "B", "B": "A"}
        )

    # Merge results
    pair_result_df = pd.merge(
        pair_result_df, pair_original_df, on=["ID_A", "ID_B"], how="left"
    )

    # Add features
    pair_result_df["setup_correct"] = (
        pair_result_df["setup_true"] == pair_result_df["setup_predicted"]
    )
    pair_result_dfs[(dataset_name, model_w_params, time_tag)] = pair_result_df

## Distributions

In [ ]:
# Density distribution of objective variables
ncol = 3
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, (dataset_name, exp_extracted_df) in enumerate(exp_extracted_dfs.items()):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.histplot(
        exp_extracted_df,
        x=obj_var,
        bins=30,
        kde=True,
        stat="density",
        alpha=0.5,
    )
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(obj_var, fontsize=10)
    plt.ylabel("Density", fontsize=10)
    plt.title(f"{dataset_name}", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Density distribution of absolute differences in objective variables between pairs
ncol = 3
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, (dataset_name, pair_original_df) in enumerate(pair_original_dfs.items()):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.histplot(
        pair_original_df,
        x=f"{obj_var}_diff_abs",
        bins=30,
        kde=True,
        stat="density",
        alpha=0.5,
    )
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"Absolute {obj_var} difference", fontsize=10)
    plt.ylabel("Density", fontsize=10)
    plt.title(f"{dataset_name}", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Density distribution of absolute differences in objective variable, stratified by correct and incorrect
ncol = len(np.unique([model_w_params for _, model_w_params, _ in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 2))
for i, ((dataset_name, model_w_params, _), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    for correct_val, color, label in zip(
        [True, False], ["tab:blue", "tab:red"], ["Correct", "Incorrect"]
    ):
        sns.histplot(
            pair_result_df.loc[
                pair_result_df["setup_correct"] == correct_val, f"{obj_var}_diff_abs"
            ],
            bins=30,
            kde=True,
            stat="density",
            common_norm=False,
            alpha=0.3,
            color=color,
            label=label,
            line_kws={"linewidth": 3, "alpha": 0.8},
        )
    plt.legend(fontsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"Absolute {obj_var} difference", fontsize=10)
    plt.ylabel("Density", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/pairwise_comparison_histogram.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/pairwise_comparison_histogram.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/pairwise_comparison_histogram.svg", format="svg", bbox_inches="tight"
)
plt.show()